In [1]:
import numpy as np
import pandas as pd
import os
import pathlib
import matplotlib.pyplot as plt
from tqdm import tqdm
from multiprocessing import Pool
from concurrent.futures import ProcessPoolExecutor
import concurrent.futures

import scipy.special as sp
import yaml
import pathlib
from pathlib import Path
import shutil

In [2]:
#--------------------------------------------------------------------------------------------------------------------------
# pdf
#--------------------------------------------------------------------------------------------------------------------------

data_name = "Default"

#--------------------------------------------------------------------------------------------------------------------------
# cut
#--------------------------------------------------------------------------------------------------------------------------

qTdQ_cut = 0.2

#--------------------------------------------------------------------------------------------------------------------------
# read path
#--------------------------------------------------------------------------------------------------------------------------

before_dir = Path(f"{data_name}/Processed")

#--------------------------------------------------------------------------------------------------------------------------
# write path
#--------------------------------------------------------------------------------------------------------------------------

after_dir = Path(f"{data_name}/Cutted")

Auxilary Functions

In [3]:
def if_satisfy_qTdQ_cut(row, qTdQ_cut):
    qT = row["qT_max"]
    Q  = row["Q_min"]
    return (qT / Q) < qTdQ_cut

DY

In [4]:
experiments =[
    'ATLAS_7',
    'ATLAS_8',
    'ATLAS_13',  
    'CDF_I',
    'CDF_II',
    'CMS_7',
    'CMS_8',
    'CMS_13',    
    'D0_I',
    'D0_II',
    'D0_II_mu',
    'E288',
    'E605',
    'E772',
    'LHCb_7',
    'LHCb_8',
    'LHCb_13',    
    'PHENIX',
    'STAR'
]
excludes = []

In [5]:
processes = ["DY","SIDIS"]

for process in processes:
    process_dir = after_dir / process
    if process_dir.exists():
        shutil.rmtree(process_dir) 
    process_dir.mkdir(parents=True, exist_ok=True)

In [6]:
for experiment in experiments:

    exp_dir = before_dir / "DY" / experiment
    file_names = [file.stem for file in sorted(exp_dir.glob("*.csv"))]
    file_paths = [exp_dir / f"{name}.csv" for name in file_names] 

    save_dir = Path(after_dir) / "DY" / experiment
    save_dir.mkdir(parents=True, exist_ok=True)

    for file_name in (f for f in file_names if f not in excludes):

        file_path = exp_dir / f"{file_name}.csv"

        df = pd.read_csv(file_path)
        
        mask = df.apply(lambda row: if_satisfy_qTdQ_cut(row, qTdQ_cut), axis=1)

        df_cut = df[mask].copy()

        destination = after_dir / "DY" / experiment / f"{file_name}.csv"
        if destination.exists():
            raise FileExistsError(f"{file_name} already exists")
        df_cut.to_csv(destination, index=False)
        
        print(f"{file_name} generated")

ATLAS7-00y10 generated
ATLAS7-10y20 generated
ATLAS7-20y24 generated
ATLAS8-00y04 generated
ATLAS8-04y08 generated
ATLAS8-08y12 generated
ATLAS8-116Q150 generated
ATLAS8-12y16 generated
ATLAS8-16y20 generated
ATLAS8-20y24 generated
ATLAS8-46Q66 generated
ATLAS13 generated
CDF1 generated
CDF2 generated
CMS7 generated
CMS8 generated
CMS13-00y04 generated
CMS13-04y08 generated
CMS13-08y12 generated
CMS13-106Q170 generated
CMS13-12y16 generated
CMS13-16y24 generated
CMS13-170Q350 generated
CMS13-350Q1000 generated
D01 generated
D02 generated
D02mu generated
E228-200-4Q5 generated
E228-200-5Q6 generated
E228-200-6Q7 generated
E228-200-7Q8 generated
E228-200-8Q9 generated
E228-300-11Q12 generated
E228-300-4Q5 generated
E228-300-5Q6 generated
E228-300-6Q7 generated
E228-300-7Q8 generated
E228-300-8Q9 generated
E228-400-11Q12 generated
E228-400-12Q13 generated
E228-400-13Q14 generated
E228-400-5Q6 generated
E228-400-6Q7 generated
E228-400-7Q8 generated
E228-400-8Q9 generated
E605-10.5Q11.5 gen

Statistics

In [7]:
points_by_experiment = {exp: 0 for exp in experiments}
for experiment in experiments:
    exp_dir = after_dir / "DY" / experiment
    file_names = [p.stem for p in sorted(exp_dir.glob("*.csv"))]

    for file_name in (f for f in file_names if f not in excludes):
        df = pd.read_csv(exp_dir / f"{file_name}.csv")

        # count points
        points_by_experiment[experiment] += len(df)

df = (pd.Series(points_by_experiment, name="points")
        .rename_axis("experiment").reset_index()
        .sort_values("points", ascending=False, ignore_index=True))
df_out = pd.concat([df, pd.DataFrame([{"experiment": "Total", "points": df["points"].sum()}])],
                   ignore_index=True)
display(df_out)

,experiment,points
0,E288,120
1,CMS_13,94
2,ATLAS_8,48
3,E605,48
4,E772,48
5,CDF_II,26
6,CDF_I,25
7,ATLAS_7,18
8,D0_I,12
9,LHCb_8,7


In [8]:
experiments = experiments =[
    'ATLAS_7',
    'ATLAS_8',
    'ATLAS_13',  
    'CDF_I',
    'CDF_II',
    'CMS_7',
    'CMS_8',
    'CMS_13',    
    'D0_I',
    'D0_II',
    'D0_II_mu',
    'E288',
    'LHCb_7',
    'LHCb_8',
    'LHCb_13',    
    'PHENIX',
    'STAR'
]

In [9]:
overall_qT_max, overall_qT_src = -np.inf, None
overall_xmax,  overall_xmax_src = -np.inf, None
overall_xmin,  overall_xmin_src =  np.inf, None
points_by_experiment = {exp: 0 for exp in experiments}

# NEW: keep all offenders and each experiment's worst offender
xmax_offenders = {}  # exp -> list[(file_name, x_max)]
worst_xmax_by_exp = {}  # exp -> (x_max, file_name)

for experiment in experiments:
    exp_dir = after_dir / "DY" / experiment
    file_names = [p.stem for p in sorted(exp_dir.glob("*.csv"))]

    for file_name in (f for f in file_names if f not in excludes):
        df = pd.read_csv(exp_dir / f"{file_name}.csv")

        # count points
        points_by_experiment[experiment] += len(df)

        # global max qT_max
        vals = df["qT_max"].to_numpy()
        i = int(np.argmax(vals))
        m = float(vals[i])
        if m > overall_qT_max:
            overall_qT_max = m
            overall_qT_src = (experiment, file_name, int(df.index[i]))

        # x bounds from extremes (largest/smallest possible x)
        sqrts = float(df["sqrts"].iat[0])  # √s
        Qmin  = float(df["Q_min"].iat[0])
        Qmax  = float(df["Q_max"].iat[0])
        y_min = float(df["y_min"].iat[0])
        y_max = float(df["y_max"].iat[0])

        xp_max = Qmax * np.exp(y_max)  / sqrts
        xp_min = Qmin * np.exp(y_min)  / sqrts
        xN_max = Qmax * np.exp(-y_min) / sqrts
        xN_min = Qmin * np.exp(-y_max) / sqrts

        x_max = float(max(xp_max, xN_max))
        x_min = float(min(xp_min, xN_min))

        if x_max > overall_xmax:
            overall_xmax = x_max
            overall_xmax_src = (experiment, file_name)
        if x_min < overall_xmin:
            overall_xmin = x_min
            overall_xmin_src = (experiment, file_name)

        # NEW: record any x_max > 1
        if x_max > 1.0:
            xmax_offenders.setdefault(experiment, []).append((file_name, x_max))
            curr_best = worst_xmax_by_exp.get(experiment)
            if (curr_best is None) or (x_max > curr_best[0]):
                worst_xmax_by_exp[experiment] = (x_max, file_name)

# report
exp, fname, idx = overall_qT_src
print(f"Global max qT_max: {overall_qT_max} (experiment={exp}, file={fname}, row_index={idx})")
print(f"Global x_max: {overall_xmax} from {overall_xmax_src}")
print(f"Global x_min: {overall_xmin} from {overall_xmin_src}\n")

# NEW: offenders report
if xmax_offenders:
    print("Experiments with x_max > 1:")
    for exp, items in xmax_offenders.items():
        worst_x, worst_file = worst_xmax_by_exp[exp][0], worst_xmax_by_exp[exp][1]
        print(f"  - {exp}: {len(items)} file(s) flagged; worst x_max={worst_x:.6g} in file {worst_file}")
        for fn, xv in items:
            print(f"      * {fn}: x_max={xv:.6g}")
else:
    print("No experiments had x_max > 1.")
 

Global max qT_max: 52.0 (experiment=CMS_13, file=CMS13-350Q1000, row_index=10)
Global x_max: 2.1341069040046157 from ('CDF_I', 'CDF1')
Global x_min: 5.1272291714964494e-05 from ('LHCb_13', 'LHCb13')

Experiments with x_max > 1:
  - CDF_I: 1 file(s) flagged; worst x_max=2.13411 in file CDF1
      * CDF1: x_max=2.13411
  - CDF_II: 1 file(s) flagged; worst x_max=1.95989 in file CDF2
      * CDF2: x_max=1.95989
  - D0_I: 1 file(s) flagged; worst x_max=1.93173 in file D01
      * D01: x_max=1.93173
  - D0_II: 1 file(s) flagged; worst x_max=1.85852 in file D02
      * D02: x_max=1.85852
  - D0_II_mu: 1 file(s) flagged; worst x_max=1.943 in file D02mu
      * D02mu: x_max=1.943
  - LHCb_7: 1 file(s) flagged; worst x_max=1.54315 in file LHCb7
      * LHCb7: x_max=1.54315
  - LHCb_8: 1 file(s) flagged; worst x_max=1.35026 in file LHCb8
      * LHCb8: x_max=1.35026
